In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
silver_df = spark.table(SILVER_TABLE)

# Gold Price Signals

In [0]:
# Calculate regional price threshold (95th percentile)
price_thresholds = (
    silver_valid_df
    .groupBy("region")
    .agg(F.expr("percentile_approx(price_rrp, 0.95)").alias("price_spike_threshold"))
)

gold_price_signals = (
    silver_valid_df.alias("s")
    .join(price_thresholds.alias("p"), on="region", how="left")
    .withColumn(
        "is_spike",
        F.when(F.col("price_rrp") >= F.col("price_spike_threshold"), F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "spike_severity",
        F.when(F.col("price_rrp") >= F.col("price_spike_threshold") * 1.5, F.lit("EXTREME"))
         .when(F.col("price_rrp") >= F.col("price_spike_threshold"), F.lit("HIGH"))
         .otherwise(F.lit("NORMAL"))
    )
    .select(
        "settlement_timestamp",
        "region",
        "price_rrp",
        "price_spike_threshold",
        "is_spike",
        "spike_severity",
        "source_file",
        "ingested_at"
    )
)

gold_price_signals.write.format("delta").mode("overwrite").saveAsTable(GOLD_SPIKE_TABLE)

# Gold Energy Demand Signals

In [0]:
# Regional 90th percentile demand threshold
demand_thresholds = (
    silver_valid_df
    .groupBy("region")
    .agg(F.expr("percentile_approx(total_demand_mw, 0.90)").alias("peak_demand_threshold"))
)

# Window for ramp calculation
demand_window = Window.partitionBy("region").orderBy("settlement_timestamp")

gold_demand_signals = (
    silver_valid_df.alias("s")
    .join(demand_thresholds.alias("d"), on="region", how="left")
    .withColumn(
        "previous_demand_mw",
        F.lag("total_demand_mw").over(demand_window)
    )
    .withColumn(
        "demand_ramp_mw",
        F.col("total_demand_mw") - F.col("previous_demand_mw")
    )
    .withColumn(
        "is_peak_demand",
        F.when(F.col("total_demand_mw") >= F.col("peak_demand_threshold"), F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "grid_stress_level",
        F.when(F.col("total_demand_mw") >= F.col("peak_demand_threshold") * 1.1, F.lit("EXTREME"))
         .when(F.col("total_demand_mw") >= F.col("peak_demand_threshold"), F.lit("HIGH"))
         .otherwise(F.lit("NORMAL"))
    )
    .select(
        "settlement_timestamp",
        "region",
        "total_demand_mw",
        "previous_demand_mw",
        "demand_ramp_mw",
        "peak_demand_threshold",
        "is_peak_demand",
        "grid_stress_level",
        "source_file",
        "ingested_at"
    )
)

gold_demand_signals.write.format("delta").mode("overwrite").saveAsTable(GOLD_STRESS_TABLE)

# Gold Battery Actions

In [0]:
# Regional price thresholds for low/high decision
battery_price_thresholds = (
    silver_valid_df
    .groupBy("region")
    .agg(
        F.expr("percentile_approx(price_rrp, 0.20)").alias("low_price_threshold"),
        F.expr("percentile_approx(price_rrp, 0.80)").alias("high_price_threshold"),
        F.expr("percentile_approx(total_demand_mw, 0.90)").alias("high_demand_threshold")
    )
)

gold_battery_actions = (
    silver_valid_df.alias("s")
    .join(battery_price_thresholds.alias("t"), on="region", how="left")
    .withColumn(
        "is_low_price",
        F.when(F.col("price_rrp") <= F.col("low_price_threshold"), F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "is_high_price",
        F.when(F.col("price_rrp") >= F.col("high_price_threshold"), F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "is_high_demand",
        F.when(F.col("total_demand_mw") >= F.col("high_demand_threshold"), F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "recommended_action",
        F.when((F.col("is_high_price") == 1) | (F.col("is_high_demand") == 1), F.lit("DISCHARGE"))
         .when(F.col("is_low_price") == 1, F.lit("CHARGE"))
         .otherwise(F.lit("IDLE"))
    )
    .withColumn(
        "action_reason",
        F.when((F.col("is_high_price") == 1) & (F.col("is_high_demand") == 1), F.lit("HIGH_PRICE_AND_HIGH_DEMAND"))
         .when(F.col("is_high_price") == 1, F.lit("HIGH_PRICE"))
         .when(F.col("is_high_demand") == 1, F.lit("HIGH_DEMAND"))
         .when(F.col("is_low_price") == 1, F.lit("LOW_PRICE"))
         .otherwise(F.lit("NORMAL_MARKET"))
    )
    .select(
        "settlement_timestamp",
        "region",
        "price_rrp",
        "total_demand_mw",
        "low_price_threshold",
        "high_price_threshold",
        "high_demand_threshold",
        "is_low_price",
        "is_high_price",
        "is_high_demand",
        "recommended_action",
        "action_reason",
        "source_file",
        "ingested_at"
    )
)

gold_battery_actions.write.format("delta").mode("overwrite").saveAsTable(GOLD_BATTERY_DISPATCH_TABLE)

# Gold Market Summary

In [0]:
# Reuse price spike threshold
market_price_thresholds = (
    silver_valid_df
    .groupBy("region")
    .agg(F.expr("percentile_approx(price_rrp, 0.95)").alias("price_spike_threshold"))
)

market_enriched_df = (
    silver_valid_df.alias("s")
    .join(market_price_thresholds.alias("p"), on="region", how="left")
    .withColumn(
        "is_spike",
        F.when(F.col("price_rrp") >= F.col("price_spike_threshold"), F.lit(1)).otherwise(F.lit(0))
    )
)

gold_market_summary = (
    market_enriched_df
    .groupBy("region", "year", "month", "day", "hour")
    .agg(
        F.avg("price_rrp").alias("avg_price_rrp"),
        F.max("price_rrp").alias("max_price_rrp"),
        F.min("price_rrp").alias("min_price_rrp"),
        F.avg("total_demand_mw").alias("avg_demand_mw"),
        F.max("total_demand_mw").alias("max_demand_mw"),
        F.sum("is_spike").alias("spike_count"),
        F.count("*").alias("interval_count")
    )
    .orderBy("region", "year", "month", "day", "hour")
)

gold_market_summary.write.format("delta").mode("overwrite").saveAsTable(GOLD_MARKET_TABLE)